# Modules and Packages

> Generated from [`05-Modules-And-Packages.py`](05-Modules-And-Packages.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
How Python organizes code across files (modules) and folders (packages) — different import styles, your own local packages, and how the import system actually behaves under the hood.

### Key points
- A **module** is a single `.py` file. A **package** is a directory of modules, historically marked with `__init__.py`.
- Import styles: `import math`, `import math as m`, `from math import sqrt`, `from math import pi as PI`.
- `from module import *` pulls all public names into the current namespace — generally discouraged since it's unclear where a name came from and risks clashes.
- `dir(module)` lists a module's contents, useful for exploring what's available.
- `__name__ == "__main__"` idiom: code under this guard only runs when the file is executed directly, not when imported — every module has a `__name__` that's `"__main__"` only when run directly.
- `__all__ = [...]` at module level explicitly controls what `import *` pulls in; anything not listed is skipped by `*` (but still reachable via a normal dotted import).
- Relative imports inside a package use leading dots: `from . import sibling_module` (current package), `from .. import x` (parent package) — only valid inside a package, never in a script run directly.
- `importlib.import_module("name")` imports a module by string name, useful when the module name is only known at runtime (config/plugin systems).
- Namespace packages (PEP 420, 3.3+): a directory **without** `__init__.py` can still act as a package, letting one logical package span multiple directories/distributions.
- Standard library (ships with Python) vs third-party (installed via `pip`) vs local (your own project code) — three sources of importable modules.

### Gotchas (the part worth revising twice)
- **Module caching via `sys.modules`**: a module's top-level code runs only the FIRST time it's imported anywhere in the program. Every later import (even via a different path/alias, or via `importlib`) reuses the same cached module object — no re-execution, no repeated side effects.
- A directly-run script's own folder is automatically added to `sys.path`, which is why a sibling package/folder is importable without extra setup — but this only applies to the *entry-point* script, not to modules imported deeper in the chain.
- `__all__` only affects `from module import *` — it does NOT make other names private or block a normal `from module import _private_thing`.

### When it matters
Any project beyond a single script needs this — splitting code into modules/packages keeps things organized, and understanding import caching/relative imports avoids confusing "why did my print statement not run again" or "attempted relative import" errors.

### One-line recap example
```python
from demo_package.math_utils import add   # explicit import of one name

print(add(2, 3))   # 5 — demo_package's __init__.py already ran once, cached in sys.modules
```

In [ ]:
import math                       # 02. import a built-in module
import math as m                  # 03. import with an alias
from math import sqrt             # 04. import a specific name
from math import pi as PI         # 05. import a specific name with an alias

## 01. Modules and Packages - Introduction

- A module is a single .py file containing code (functions, classes, vars).
- A package is a directory of modules, marked with an __init__.py file.
- Splitting code this way keeps large programs organized and reusable.

In [ ]:
# 02-05. Different Import Styles (see imports at top of file)
# ------------------------------------
print("\n# 02-05. Different Import Styles")
print("math.sqrt(16):", math.sqrt(16))
print("m.pi (aliased module):", m.pi)
print("sqrt(25) (imported name directly):", sqrt(25))
print("PI (imported + aliased name):", PI)

## 06. import * - Import Everything (generally discouraged)

## 07. dir() - Inspecting a Module's Contents

In [ ]:
print([name for name in dir(math) if not name.startswith("_")][:10], "...")

## 08. The __name__ == "__main__" Idiom

- Every module has a __name__ variable.
- When run directly, __name__ == "__main__".
- When imported by another file, __name__ == the module's own name.

In [ ]:
print("this file's __name__:", __name__)
print("# - Code inside `if __name__ == \"__main__\":` only runs when the")
print("#   file is executed directly, not when it's imported elsewhere.")

## 09. Creating and Importing Your Own Module

- See demo_package/helper.py and demo_package/math_utils.py next to this file.
- Python automatically adds a directly-run script's own folder to sys.path,
  so the sibling demo_package/ is importable without any extra setup.

In [ ]:
from demo_package import helper
from demo_package.math_utils import add, multiply

print("\n# 09. Creating and Importing Your Own Module")
print(helper.GREETING)
print("helper.shout('hi'):", helper.shout("hi"))
print("add(2, 3):", add(2, 3))
print("multiply(4, 5):", multiply(4, 5))

## 10. Package Structure (reference)

## 11. Standard Library, Third-Party, and Local Modules

## 12. Virtual Environments and pip (reference, not executed)

## 13. __all__ - Controlling `from module import *`

- __all__ is a list of names, defined at module level, that explicitly
  controls what `import *` pulls in. Anything not listed is skipped by
  the star-import (though still reachable via a normal dotted import).
- See demo_package/extra.py: __all__ = ["public_function", "PUBLIC_CONSTANT"]

In [ ]:
from demo_package.extra import *  # only names in __all__ come in

print("public_function() available:", public_function())
print("PUBLIC_CONSTANT available:", PUBLIC_CONSTANT)
print("_private_function was NOT imported by *:", "_private_function" not in dir())

import demo_package.extra as extra_module
print("but it's still reachable via the module itself:", extra_module._private_function())

## 14. Relative Imports Within a Package (conceptual)

- Inside a package, modules can import siblings with a LEADING DOT instead
  of the full dotted path. demo_package is flat here (no sub-packages),
  so this is explained rather than executed - the syntax still matters.

## 15. importlib.import_module() - Dynamic Imports

- Sometimes the module name is only known at runtime (e.g. from a config
  file or plugin system). importlib.import_module() imports by STRING name.

In [ ]:
import importlib

print("\n# 15. importlib.import_module() for Dynamic Imports")
module_name = "math"  # could come from user input / config
dynamic_math = importlib.import_module(module_name)
print(f"dynamically imported '{module_name}', dynamic_math.sqrt(9):", dynamic_math.sqrt(9))

## 16. Module Caching via sys.modules

- Python only RUNS a module's top-level code the first time it's imported.
  Every import after that (from anywhere in the program) reuses the same
  cached module object from sys.modules - no re-execution, no re-print.

In [ ]:
import sys

print("\n# 16. Module Caching via sys.modules")
print("demo_package.extra already in sys.modules:", "demo_package.extra" in sys.modules)
print("importing it again - no 'module body is executing' print below:")
import demo_package.extra  # already cached, body does NOT run again
importlib.import_module("demo_package.extra")  # same story via importlib
print("(confirmed: no side-effect print appeared above this line on re-import)")

## 17. Namespace Packages (PEP 420, brief mention)

- Since Python 3.3, a directory WITHOUT __init__.py can still act as a
  package (a "namespace package"), letting a single logical package be
  split across multiple directories/distributions on disk.
- demo_package has an __init__.py (a "regular" package) - namespace
  packages are the __init__.py-less alternative for that specific use case.